# Tox21 + ZINC Improved Training Notebook

This notebook implements the current project strategy without changing scope:

- `tox21.csv` remains the only supervised toxicity label source
- `250k_rndm_zinc_drugs_clean_3.csv` is used as an unlabeled chemistry-space support dataset
- per-assay binary classification is preserved
- `RandomForestClassifier` and `XGBClassifier` are the only primary model families
- validation now searches a small set of RF/XGB configurations per assay
- blend weights are learned on the validation set instead of being fixed at 0.5 / 0.5
- `KMeans` cluster-based splitting is preserved
- per-assay threshold tuning is applied on the validation set


In [ ]:
%pip install -q rdkit xgboost imbalanced-learn shap tqdm

In [ ]:
from pathlib import Path
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Lipinski, QED, rdMolDescriptors

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

In [ ]:
# Paths for Colab. Upload the CSV files to /content or mount Drive and change DATA_DIR.
DATA_DIR = Path('/content')
TOX21_PATH = DATA_DIR / 'tox21.csv'
ZINC_PATH = DATA_DIR / '250k_rndm_zinc_drugs_clean_3.csv'
OUTPUT_DIR = DATA_DIR / 'outputs_improved'
MODEL_DIR = DATA_DIR / 'models_improved'
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

TARGETS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

FP_BITS = 1024
USE_SMOTE = False
SMOTE_K_NEIGHBORS = 5
ZINC_SAMPLE_SIZE = 25000
KMEANS_CLUSTERS = 20
NN_NEIGHBORS = 5
RANDOM_STATE = 42
RF_CONFIGS = [
    {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 2, 'max_features': 'sqrt'},
    {'n_estimators': 700, 'max_depth': 18, 'min_samples_leaf': 1, 'max_features': 'sqrt'},
    {'n_estimators': 500, 'max_depth': 24, 'min_samples_leaf': 2, 'max_features': 0.35},
]
XGB_CONFIGS = [
    {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.90, 'colsample_bytree': 0.80, 'min_child_weight': 3, 'reg_alpha': 0.0, 'reg_lambda': 1.0},
    {'n_estimators': 900, 'max_depth': 4, 'learning_rate': 0.03, 'subsample': 0.85, 'colsample_bytree': 0.75, 'min_child_weight': 2, 'reg_alpha': 0.0, 'reg_lambda': 1.5},
    {'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.04, 'subsample': 0.90, 'colsample_bytree': 0.85, 'min_child_weight': 5, 'reg_alpha': 0.0, 'reg_lambda': 1.0},
]
BLEND_WEIGHT_GRID = np.linspace(0.0, 1.0, 21)

HAS_GPU = os.path.exists('/proc/driver/nvidia/version')
XGB_DEVICE = 'cuda' if HAS_GPU else 'cpu'
print({'HAS_GPU': HAS_GPU, 'XGB_DEVICE': XGB_DEVICE})

In [ ]:
tox21 = pd.read_csv(TOX21_PATH)
zinc = pd.read_csv(ZINC_PATH)

tox21['smiles'] = tox21['smiles'].astype(str).str.strip()
zinc['smiles'] = zinc['smiles'].astype(str).str.replace('\n', '', regex=False).str.strip()

zinc_sample = zinc.sample(n=min(ZINC_SAMPLE_SIZE, len(zinc)), random_state=RANDOM_STATE).reset_index(drop=True)

print('tox21 shape:', tox21.shape)
print('zinc shape:', zinc.shape)
print('zinc_sample shape:', zinc_sample.shape)

In [ ]:
def mol_from_smiles(smiles: str):
    return Chem.MolFromSmiles(smiles)

def morgan_fp(mol, radius=2, n_bits=FP_BITS):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def calc_descriptor_dict(mol):
    return {
        'MolWt': Descriptors.MolWt(mol),
        'ExactMolWt': Descriptors.ExactMolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'MolMR': Descriptors.MolMR(mol),
        'TPSA': rdMolDescriptors.CalcTPSA(mol),
        'LabuteASA': rdMolDescriptors.CalcLabuteASA(mol),
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        'RotBonds': Lipinski.NumRotatableBonds(mol),
        'RingCount': Lipinski.RingCount(mol),
        'AromaticRings': Lipinski.NumAromaticRings(mol),
        'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol),
        'HeavyAtomCount': Lipinski.HeavyAtomCount(mol),
        'HeteroAtoms': Lipinski.NumHeteroatoms(mol),
        'ValenceElectrons': Descriptors.NumValenceElectrons(mol),
        'NHOHCount': Lipinski.NHOHCount(mol),
        'NOCount': Lipinski.NOCount(mol),
        'NumAliphaticRings': Lipinski.NumAliphaticRings(mol),
        'NumAromaticHeterocycles': Lipinski.NumAromaticHeterocycles(mol),
        'NumSaturatedRings': Lipinski.NumSaturatedRings(mol),
        'NumBridgeheadAtoms': rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        'NumSpiroAtoms': rdMolDescriptors.CalcNumSpiroAtoms(mol),
        'BertzCT': Descriptors.BertzCT(mol),
        'HallKierAlpha': Descriptors.HallKierAlpha(mol),
        'BalabanJ': Descriptors.BalabanJ(mol),
        'QED': QED.qed(mol),
    }

def featurize_dataframe(df, include_fingerprints=True):
    valid_idx = []
    desc_rows = []
    fp_rows = []

    for idx, smiles in df['smiles'].items():
        mol = mol_from_smiles(smiles)
        if mol is None:
            continue
        valid_idx.append(idx)
        desc_rows.append(calc_descriptor_dict(mol))
        if include_fingerprints:
            fp_rows.append(morgan_fp(mol))

    desc_df = pd.DataFrame(desc_rows, index=valid_idx)
    out_df = df.loc[valid_idx].copy()
    if include_fingerprints:
        fp_df = pd.DataFrame(fp_rows, index=valid_idx, columns=[f'fp_{i}' for i in range(FP_BITS)])
        return out_df, desc_df, fp_df
    return out_df, desc_df, None

tox21_valid, tox21_desc, tox21_fp = featurize_dataframe(tox21, include_fingerprints=True)
zinc_valid, zinc_desc, _ = featurize_dataframe(zinc_sample, include_fingerprints=False)

print('valid tox21:', tox21_valid.shape, tox21_desc.shape, tox21_fp.shape)
print('valid zinc sample:', zinc_valid.shape, zinc_desc.shape)

In [ ]:
# Add ZINC-provided properties for chemistry-space support features.
zinc_valid[['logP', 'qed', 'SAS']] = zinc_valid[['logP', 'qed', 'SAS']].apply(pd.to_numeric, errors='coerce')

descriptor_cols = list(tox21_desc.columns)
space_cols = descriptor_cols.copy()

scaler = StandardScaler()
combined_space = pd.concat([
    tox21_desc[space_cols],
    zinc_desc[space_cols]
], axis=0)
combined_scaled = scaler.fit_transform(combined_space)

kmeans = KMeans(n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(combined_scaled)

tox21_scaled = scaler.transform(tox21_desc[space_cols])
zinc_scaled = scaler.transform(zinc_desc[space_cols])

tox21_clusters = pd.Series(kmeans.predict(tox21_scaled), index=tox21_desc.index, name='cluster')
tox21_cluster_distance = pd.Series(kmeans.transform(tox21_scaled).min(axis=1), index=tox21_desc.index, name='cluster_distance')

nn = NearestNeighbors(n_neighbors=NN_NEIGHBORS, metric='euclidean')
nn.fit(zinc_scaled)
distances, neighbors = nn.kneighbors(tox21_scaled)

neighbor_logP = zinc_valid.iloc[neighbors.flatten()]['logP'].to_numpy().reshape(len(neighbors), NN_NEIGHBORS)
neighbor_qed = zinc_valid.iloc[neighbors.flatten()]['qed'].to_numpy().reshape(len(neighbors), NN_NEIGHBORS)
neighbor_SAS = zinc_valid.iloc[neighbors.flatten()]['SAS'].to_numpy().reshape(len(neighbors), NN_NEIGHBORS)

tox21_support = pd.DataFrame({
    'zinc_nn_dist_min': distances.min(axis=1),
    'zinc_nn_dist_mean': distances.mean(axis=1),
    'zinc_nn_dist_std': distances.std(axis=1),
    'zinc_local_density': 1.0 / (1.0 + distances.mean(axis=1)),
    'zinc_nn_logP_mean': neighbor_logP.mean(axis=1),
    'zinc_nn_logP_std': neighbor_logP.std(axis=1),
    'zinc_nn_qed_mean': neighbor_qed.mean(axis=1),
    'zinc_nn_qed_std': neighbor_qed.std(axis=1),
    'zinc_nn_SAS_mean': neighbor_SAS.mean(axis=1),
    'zinc_nn_SAS_std': neighbor_SAS.std(axis=1),
    'cluster_distance': tox21_cluster_distance.values,
}, index=tox21_desc.index)

X = pd.concat([tox21_desc, tox21_support, tox21_fp], axis=1)
print('final feature matrix shape:', X.shape)
tox21_support.head()

In [ ]:
# KMeans cluster-based split on Tox21 indices only.
cluster_ids = sorted(tox21_clusters.unique())
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(cluster_ids)

train_cut = int(0.70 * len(cluster_ids))
val_cut = int(0.85 * len(cluster_ids))

train_clusters = set(cluster_ids[:train_cut])
val_clusters = set(cluster_ids[train_cut:val_cut])
test_clusters = set(cluster_ids[val_cut:])

split_series = tox21_clusters.map(lambda c: 'train' if c in train_clusters else ('val' if c in val_clusters else 'test'))
split_series.name = 'split'
split_series.value_counts()

In [ ]:
def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc': float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
    }

def metric_rank(metrics):
    return (
        float(np.nan_to_num(metrics['pr_auc'], nan=-1.0)),
        float(np.nan_to_num(metrics['roc_auc'], nan=-1.0)),
        float(np.nan_to_num(metrics['f1'], nan=-1.0)),
    )

def choose_threshold(y_true, y_prob, min_precision=0.15, beta=1.5):
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    best = {'threshold': 0.5, 'f1': -1.0, 'precision': 0.0, 'recall': 0.0, 'fbeta': -1.0}
    for threshold in np.linspace(0.01, 0.99, 99):
        y_pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        denom = (beta ** 2 * precision) + recall
        fbeta = 0.0 if denom == 0 else ((1 + beta ** 2) * precision * recall) / denom
        if precision >= min_precision and (fbeta, f1, recall) > (best['fbeta'], best['f1'], best['recall']):
            best = {'threshold': float(threshold), 'f1': float(f1), 'precision': float(precision), 'recall': float(recall), 'fbeta': float(fbeta)}
    if best['fbeta'] < 0:
        best_idx = int(np.argmax((2 * precision_curve * recall_curve) / np.clip(precision_curve + recall_curve, 1e-12, None)))
        fallback_threshold = 0.30 if best_idx == 0 else float(np.linspace(0.01, 0.99, 99)[min(best_idx - 1, 98)])
        y_pred = (y_prob >= fallback_threshold).astype(int)
        best = {
            'threshold': float(fallback_threshold),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'fbeta': 0.0,
        }
    return best

def maybe_apply_smote(X_train, y_train):
    if not USE_SMOTE:
        return X_train, y_train
    if y_train.nunique() < 2 or y_train.sum() <= SMOTE_K_NEIGHBORS:
        return X_train, y_train
    sampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    return X_res, y_res

def build_rf(config):
    return RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        min_samples_leaf=config['min_samples_leaf'],
        max_features=config['max_features'],
        n_jobs=-1,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
    )

def build_xgb(config, scale_pos_weight):
    return XGBClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        learning_rate=config['learning_rate'],
        subsample=config['subsample'],
        colsample_bytree=config['colsample_bytree'],
        min_child_weight=config['min_child_weight'],
        reg_alpha=config['reg_alpha'],
        reg_lambda=config['reg_lambda'],
        scale_pos_weight=float(scale_pos_weight),
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        tree_method='hist',
        device=XGB_DEVICE,
        n_jobs=2,
        verbosity=0,
    )

def optimize_blend(y_true, rf_prob, xgb_prob):
    best = None
    for weight_rf in BLEND_WEIGHT_GRID:
        blend_prob = weight_rf * rf_prob + (1.0 - weight_rf) * xgb_prob
        metrics = evaluate_binary(y_true, blend_prob)
        candidate = {
            'weight_rf': float(weight_rf),
            'weight_xgb': float(1.0 - weight_rf),
            'val_prob': blend_prob,
            'val_metrics': metrics,
        }
        if best is None or metric_rank(candidate['val_metrics']) > metric_rank(best['val_metrics']):
            best = candidate
    return best

In [ ]:
results = []
artifacts = {
    'targets': TARGETS,
    'feature_columns': list(X.columns),
    'descriptor_columns': descriptor_cols,
    'support_feature_columns': list(tox21_support.columns),
    'fp_bits': FP_BITS,
    'split_strategy': 'kmeans_cluster_split_70_15_15',
    'zinc_support_sample_size': int(len(zinc_valid)),
    'rf_configs': RF_CONFIGS,
    'xgb_configs': XGB_CONFIGS,
    'models': {}
}

for target in TARGETS:
    work_df = tox21_valid[[target]].join(split_series).dropna(subset=[target]).copy()
    work_df[target] = work_df[target].astype(int)

    idx_train = work_df.index[work_df['split'] == 'train']
    idx_val = work_df.index[work_df['split'] == 'val']
    idx_test = work_df.index[work_df['split'] == 'test']

    X_train = X.loc[idx_train]
    X_val = X.loc[idx_val]
    X_test = X.loc[idx_test]
    y_train = work_df.loc[idx_train, target]
    y_val = work_df.loc[idx_val, target]
    y_test = work_df.loc[idx_test, target]

    X_train_fit, y_train_fit = maybe_apply_smote(X_train, y_train)

    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    rf_runs = []
    for rf_config in RF_CONFIGS:
        rf_model = build_rf(rf_config)
        rf_model.fit(X_train_fit, y_train_fit)
        rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
        rf_runs.append({
            'model': rf_model,
            'config': rf_config,
            'val_prob': rf_val_prob,
            'val_metrics': evaluate_binary(y_val, rf_val_prob),
        })
    best_rf = max(rf_runs, key=lambda run: metric_rank(run['val_metrics']))

    xgb_runs = []
    for xgb_config in XGB_CONFIGS:
        xgb_model = build_xgb(xgb_config, scale_pos_weight)
        xgb_model.fit(X_train_fit, y_train_fit)
        xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]
        xgb_runs.append({
            'model': xgb_model,
            'config': xgb_config,
            'val_prob': xgb_val_prob,
            'val_metrics': evaluate_binary(y_val, xgb_val_prob),
        })
    best_xgb = max(xgb_runs, key=lambda run: metric_rank(run['val_metrics']))

    blend_best = optimize_blend(y_val, best_rf['val_prob'], best_xgb['val_prob'])

    candidates = {
        'random_forest': (best_rf['val_metrics'], best_rf['val_prob']),
        'xgboost': (best_xgb['val_metrics'], best_xgb['val_prob']),
        'blend': (blend_best['val_metrics'], blend_best['val_prob']),
    }
    selected_name = max(candidates.keys(), key=lambda name: metric_rank(candidates[name][0]))
    selected_val_prob = candidates[selected_name][1]
    threshold_info = choose_threshold(y_val, selected_val_prob, min_precision=0.15)
    threshold = threshold_info['threshold']

    rf_test_prob = best_rf['model'].predict_proba(X_test)[:, 1]
    xgb_test_prob = best_xgb['model'].predict_proba(X_test)[:, 1]
    blend_test_prob = blend_best['weight_rf'] * rf_test_prob + blend_best['weight_xgb'] * xgb_test_prob
    test_prob_map = {
        'random_forest': rf_test_prob,
        'xgboost': xgb_test_prob,
        'blend': blend_test_prob,
    }
    test_metrics = evaluate_binary(y_test, test_prob_map[selected_name], threshold=threshold)

    rf_path = MODEL_DIR / f'{target}__rf.joblib'
    xgb_path = MODEL_DIR / f'{target}__xgb.joblib'
    joblib.dump(best_rf['model'], rf_path)
    joblib.dump(best_xgb['model'], xgb_path)

    artifacts['models'][target] = {
        'selected_model': selected_name,
        'threshold': float(threshold),
        'rf_model_path': str(rf_path),
        'xgb_model_path': str(xgb_path),
        'rf_config': best_rf['config'],
        'xgb_config': best_xgb['config'],
        'blend_weights': [float(blend_best['weight_rf']), float(blend_best['weight_xgb'])],
        'scale_pos_weight': float(scale_pos_weight),
    }

    results.append({
        'target': target,
        'train_n': int(len(idx_train)),
        'val_n': int(len(idx_val)),
        'test_n': int(len(idx_test)),
        'positive_rate_train': float(y_train.mean()),
        'rf_val_roc_auc': best_rf['val_metrics']['roc_auc'],
        'rf_val_pr_auc': best_rf['val_metrics']['pr_auc'],
        'xgb_val_roc_auc': best_xgb['val_metrics']['roc_auc'],
        'xgb_val_pr_auc': best_xgb['val_metrics']['pr_auc'],
        'blend_val_roc_auc': blend_best['val_metrics']['roc_auc'],
        'blend_val_pr_auc': blend_best['val_metrics']['pr_auc'],
        'selected_model': selected_name,
        'selected_threshold': float(threshold),
        'blend_weight_rf': float(blend_best['weight_rf']),
        'blend_weight_xgb': float(blend_best['weight_xgb']),
        'test_roc_auc': test_metrics['roc_auc'],
        'test_pr_auc': test_metrics['pr_auc'],
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
    })

results_df = pd.DataFrame(results).sort_values(['test_pr_auc', 'test_roc_auc'], ascending=False)
results_df

In [ ]:
results_path = OUTPUT_DIR / 'tox21_zinc_improved_results.csv'
metadata_path = MODEL_DIR / 'metadata.json'

results_df.to_csv(results_path, index=False)
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(artifacts, f, indent=2)

print('saved results:', results_path)
print('saved metadata:', metadata_path)
results_df[['target', 'selected_model', 'selected_threshold', 'test_roc_auc', 'test_pr_auc', 'test_f1', 'test_recall']].head(12)

In [ ]:
# Optional: score a small ZINC preview with the selected assay logic.
preview_n = 10
zinc_preview = zinc_valid.head(preview_n).copy()
zinc_preview_desc = zinc_desc.loc[zinc_preview.index, descriptor_cols].copy()

preview_scaled = scaler.transform(zinc_preview_desc[space_cols])
preview_distances, preview_neighbors = nn.kneighbors(preview_scaled)
preview_neighbor_logP = zinc_valid.iloc[preview_neighbors.flatten()]['logP'].to_numpy().reshape(len(preview_neighbors), NN_NEIGHBORS)
preview_neighbor_qed = zinc_valid.iloc[preview_neighbors.flatten()]['qed'].to_numpy().reshape(len(preview_neighbors), NN_NEIGHBORS)
preview_neighbor_SAS = zinc_valid.iloc[preview_neighbors.flatten()]['SAS'].to_numpy().reshape(len(preview_neighbors), NN_NEIGHBORS)
preview_cluster_distance = kmeans.transform(preview_scaled).min(axis=1)

preview_support = pd.DataFrame({
    'zinc_nn_dist_min': preview_distances.min(axis=1),
    'zinc_nn_dist_mean': preview_distances.mean(axis=1),
    'zinc_nn_dist_std': preview_distances.std(axis=1),
    'zinc_local_density': 1.0 / (1.0 + preview_distances.mean(axis=1)),
    'zinc_nn_logP_mean': preview_neighbor_logP.mean(axis=1),
    'zinc_nn_logP_std': preview_neighbor_logP.std(axis=1),
    'zinc_nn_qed_mean': preview_neighbor_qed.mean(axis=1),
    'zinc_nn_qed_std': preview_neighbor_qed.std(axis=1),
    'zinc_nn_SAS_mean': preview_neighbor_SAS.mean(axis=1),
    'zinc_nn_SAS_std': preview_neighbor_SAS.std(axis=1),
    'cluster_distance': preview_cluster_distance,
}, index=zinc_preview.index)

zinc_preview_fp_rows = []
for smiles in zinc_preview['smiles']:
    mol = mol_from_smiles(smiles)
    zinc_preview_fp_rows.append(morgan_fp(mol))
zinc_preview_fp = pd.DataFrame(zinc_preview_fp_rows, index=zinc_preview.index, columns=[f'fp_{i}' for i in range(FP_BITS)])

X_preview = pd.concat([zinc_preview_desc, preview_support, zinc_preview_fp], axis=1)[X.columns]

for target in TARGETS:
    model_info = artifacts['models'][target]
    rf = joblib.load(model_info['rf_model_path'])
    xgb = joblib.load(model_info['xgb_model_path'])
    rf_prob = rf.predict_proba(X_preview)[:, 1]
    xgb_prob = xgb.predict_proba(X_preview)[:, 1]
    if model_info['selected_model'] == 'random_forest':
        zinc_preview[target] = rf_prob
    elif model_info['selected_model'] == 'xgboost':
        zinc_preview[target] = xgb_prob
    else:
        w_rf, w_xgb = model_info['blend_weights']
        zinc_preview[target] = w_rf * rf_prob + w_xgb * xgb_prob

zinc_preview['overall_toxicity_risk'] = zinc_preview[TARGETS].mean(axis=1)
zinc_preview[['smiles', 'logP', 'qed', 'SAS', 'overall_toxicity_risk']].head(preview_n)

## Practical notes

- This notebook stays inside the current project plan.
- ZINC is used only as an unlabeled chemistry-space support dataset and for candidate scoring.
- The selected model per assay can be `random_forest`, `xgboost`, or `blend`.
- The app should later load the saved artifacts and use the same feature pipeline.
